# Nested Learning for CAD — kiểm chứng tối ưu NN-write v5.1

Patch execution riêng, giữ đầy đủ Nested Learning. **Chưa biết speedup T4; chưa được migrate benchmark.** Đọc `docs/main_spec/throughput_nn_write_v5_1.md` và `nn_write_v5_1_implementation.md`.

**4 inputs:** newsource `hope_cad_support_v5_1_nn_write_source_r1.zip` (Kaggle giải nén cũng được), **original reviewed NESTED_seed0 benchmark376/3599** evidence (ZIP/output), MVTec raw, lockedViT-B/8 safetensors. Tháo source profiler cũ và completed profiler evidence: G1receipt đã bind trong source; parent vẫn là376, không lấy state195 của clone. Không attach W6/pilot learned states.

**GPU T4×2; Save Version → Save & Run All; kernel mới.** Giữ PHASE='validate'. R1→O1→O2→R2, mỗi pass từ179 qua16ảnh179–194.64attempts tối đa/90min cumulative,180s exportreserve. Budget không phải ETA; hai referencepasses riêng trước đây ~34phút, tổng phụ thuộc candidate/diagnostics. Không tự tăng budget hoặc retry partialpass.

Primarytiming loại các khoảng capture giống nhau ở R/O. Sau đó recompute actual/meta gradients/steps trên clone để kiểm exact; không thêmimagearrivals hoặc ADtest. Heartbeat30s và progress mỗi8arrivals. Check `summary.json`: complete không tự là speedgatePASS, không là approval adopt/migrate.

Output `hope_cad_support_v5_1_nn_write_evidence.zip` và log lưu vào `.agents/artifacts/runs/support_v5_1_nn_write_validation_01/session_01/`. Giữ mọi failure/incomplete evidence. Chỉ resume latest **NN-write validation** output khi được hướng dẫn giữa completedpasses; tháo output cũ. Sau run cần review trước khi đổi source benchmark376.


In [ ]:
PHASE='validate'
SOURCE_OVERRIDE=None
PARENT_OVERRIDE=None  # exact reviewed benchmark376 ZIP or extracted evidence root
RESUME_OVERRIDE=None  # latest profiler output only; first run None
MVTEC_OVERRIDE=None
CHECKPOINT_OVERRIDE=None

from pathlib import Path,PurePosixPath
import hashlib,json,zipfile,time,os,sys,shutil
STARTED=time.monotonic()
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
sys.dont_write_bytecode=True
INPUT=Path('/kaggle/input');ROOT=Path('/tmp/hope_nn_write_source_r1')
WORK=Path('/kaggle/working/hope_nn_write_validation')
ASSETS=Path('/tmp/hope_throughput_assets')
OUTPUT=Path('/kaggle/working/hope_cad_support_v5_1_nn_write_evidence.zip')
PINNED_CONTRACT='a06143f04a353a8e01ec295a2103c70aaa0886b0dce059381254a0c81f59793e'
print('[NN BOOTSTRAP] locating source; elapsed',round(time.monotonic()-STARTED,1),'s',flush=True)
def safe(root,name):
    p=PurePosixPath(name)
    if p.is_absolute() or p.as_posix()!=name or '..' in p.parts or '\\' in name or ':' in name:raise ValueError('Unsafe source path')
    out=root/name
    if not out.resolve().is_relative_to(root.resolve()):raise ValueError('Escaping source path')
    return out
def manifest(p):
    if p.is_dir():return json.loads((p/'nn_write_manifest.json').read_text())
    with zipfile.ZipFile(p) as z:return json.loads(z.read('nn_write_manifest.json'))
choices=[Path(SOURCE_OVERRIDE)] if SOURCE_OVERRIDE else [p.parent for p in INPUT.rglob('nn_write_manifest.json')]+list(INPUT.rglob('*.zip'))
valid=[]
for p in choices:
    try:
        m=manifest(p)
        if m.get('schema')=='hope_support_v5_1_nn_write_source_v1':valid.append((p,m))
    except (OSError,KeyError,ValueError,zipfile.BadZipFile):
        if SOURCE_OVERRIDE:raise
identities={json.dumps(m,sort_keys=True) for p,m in valid}
if len(identities)!=1:raise ValueError('Expected ONE profiler source identity; set SOURCE_OVERRIDE')
source,m=sorted(valid,key=lambda v:str(v[0]))[0]
if m['contract_sha256']!=PINNED_CONTRACT:raise ValueError('Notebook/source contract version mismatch')
ROOT.mkdir(parents=True,exist_ok=True)
z=None if source.is_dir() else zipfile.ZipFile(source)
try:
    if z and (len(z.namelist())!=len(set(z.namelist())) or set(z.namelist())!=set(m['files'])|{'nn_write_manifest.json'}):raise ValueError('Source ZIP inventory mismatch')
    for name,expected in m['files'].items():
        original=safe(source,name) if source.is_dir() else None
        if original is not None and original.is_symlink():raise ValueError('Source symlink')
        raw=original.read_bytes() if original is not None else z.read(name)
        if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Source payload SHA mismatch: '+name)
        dest=safe(ROOT,name);dest.parent.mkdir(parents=True,exist_ok=True)
        if dest.exists() and dest.read_bytes()!=raw:raise ValueError('Existing source differs; use a fresh kernel/session')
        dest.write_bytes(raw)
    target=ROOT/'nn_write_manifest.json';raw=json.dumps(m,sort_keys=True).encode()
    if target.exists() and json.loads(target.read_bytes())!=m:raise ValueError('Existing delivery differs')
    target.write_bytes(raw)
finally:
    if z:z.close()
for name in ('training','models','dataset','notebook_support'):
    module=sys.modules.get(name)
    if module:
        paths=list(getattr(module,'__path__',[]))
        if not paths or any(not Path(p).resolve().is_relative_to(ROOT) for p in paths):raise ValueError('Mixed source imports; restart kernel')
sys.path.insert(0,str(ROOT));os.chdir(ROOT)
print('[NN BOOTSTRAP] verified',len(m['files']),'files;',m['origin'],flush=True)
print('Fixed90min; R1/O1/O2/R2 on16known normals; exact gradient diagnostics; no benchmark adoption',flush=True)


In [ ]:
from notebook_support.support_nn_write_v5_1.entry import run_notebook
RESULT=run_notebook(dict(phase=PHASE,inputs=str(INPUT),work=str(WORK),assets=str(ASSETS),output=str(OUTPUT),started=STARTED,
    PARENT_OVERRIDE=PARENT_OVERRIDE,RESUME_OVERRIDE=RESUME_OVERRIDE,MVTEC_OVERRIDE=MVTEC_OVERRIDE,CHECKPOINT_OVERRIDE=CHECKPOINT_OVERRIDE))
